# Titanic Dashboard & Interactive Visualizations

This notebook creates comprehensive visualizations for the Titanic analysis suitable for dashboard presentation.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
colors_palette = {'Survived': '#2ecc71', 'Not Survived': '#e74c3c'}

print("Libraries imported successfully!")

---
## Load & Prepare Data

In [ ]:
# Load data
df = pd.read_csv('train.csv')
print(f"Dataset loaded: {df.shape}")

# Add class names for better visualization
class_names = {1: 'First Class', 2: 'Second Class', 3: 'Third Class'}
df['Class_Name'] = df['Pclass'].map(class_names)

port_names = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
df['Port_Name'] = df['Embarked'].map(port_names)

df['Survived_Label'] = df['Survived'].map({0: 'Not Survived', 1: 'Survived'})
df['Gender'] = df['Sex'].map({'male': 'Male', 'female': 'Female'})

print("Data prepared for visualization")

---
## Dashboard Visualizations

### 1. Survival by Gender (Interactive Bar Chart)

In [ ]:
# Survival by Gender
gender_survival = df.groupby('Gender')['Survived'].agg(['sum', 'count'])
gender_survival['Death'] = gender_survival['count'] - gender_survival['sum']
gender_survival = gender_survival.drop('sum', axis=1).rename(columns={'count': 'Total'})

gender_data = pd.DataFrame([
    {'Gender': 'Male', 'Status': 'Survived', 'Count': len(df[(df['Gender']=='Male') & (df['Survived']==1)])},
    {'Gender': 'Male', 'Status': 'Did Not Survive', 'Count': len(df[(df['Gender']=='Male') & (df['Survived']==0)])},
    {'Gender': 'Female', 'Status': 'Survived', 'Count': len(df[(df['Gender']=='Female') & (df['Survived']==1)])},
    {'Gender': 'Female', 'Status': 'Did Not Survive', 'Count': len(df[(df['Gender']=='Female') & (df['Survived']==0)])},
])

fig = px.bar(gender_data, x='Gender', y='Count', color='Status', 
             title='Survival by Gender', barmode='group',
             color_discrete_map={'Survived': '#2ecc71', 'Did Not Survive': '#e74c3c'},
             labels={'Count': 'Number of Passengers'})
fig.update_layout(hovermode='x unified', height=500)
fig.write_html('01_survival_by_gender.html')
fig.show()
print("Saved: 01_survival_by_gender.html")

### 2. Survival by Passenger Class (Interactive Bar Chart)

In [ ]:
# Survival by Class
class_data = pd.DataFrame([
    {'Class': 'First Class', 'Status': 'Survived', 'Count': len(df[(df['Pclass']==1) & (df['Survived']==1)])},
    {'Class': 'First Class', 'Status': 'Did Not Survive', 'Count': len(df[(df['Pclass']==1) & (df['Survived']==0)])},
    {'Class': 'Second Class', 'Status': 'Survived', 'Count': len(df[(df['Pclass']==2) & (df['Survived']==1)])},
    {'Class': 'Second Class', 'Status': 'Did Not Survive', 'Count': len(df[(df['Pclass']==2) & (df['Survived']==0)])},
    {'Class': 'Third Class', 'Status': 'Survived', 'Count': len(df[(df['Pclass']==3) & (df['Survived']==1)])},
    {'Class': 'Third Class', 'Status': 'Did Not Survive', 'Count': len(df[(df['Pclass']==3) & (df['Survived']==0)])},
])

fig = px.bar(class_data, x='Class', y='Count', color='Status',
             title='Survival by Passenger Class', barmode='group',
             color_discrete_map={'Survived': '#2ecc71', 'Did Not Survive': '#e74c3c'},
             labels={'Count': 'Number of Passengers'},
             category_orders={'Class': ['First Class', 'Second Class', 'Third Class']})
fig.update_layout(hovermode='x unified', height=500)
fig.write_html('02_survival_by_class.html')
fig.show()
print("Saved: 02_survival_by_class.html")

### 3. Age Distribution (Interactive Histogram)

In [ ]:
# Age distribution by survival
fig = px.histogram(df, x='Age', color='Survived_Label', nbins=30,
                   title='Age Distribution by Survival Status',
                   labels={'Age': 'Age (years)', 'count': 'Number of Passengers'},
                   color_discrete_map={'Survived': '#2ecc71', 'Not Survived': '#e74c3c'},
                   barmode='group')
fig.update_layout(height=500, hovermode='x unified')
fig.write_html('03_age_distribution.html')
fig.show()
print("Saved: 03_age_distribution.html")

### 4. Fare Distribution (Interactive Box Plot)

In [ ]:
# Fare distribution by class and survival
fig = px.box(df, x='Class_Name', y='Fare', color='Survived_Label',
             title='Fare Distribution by Class and Survival',
             labels={'Fare': 'Ticket Fare (£)', 'Class_Name': 'Passenger Class'},
             color_discrete_map={'Survived': '#2ecc71', 'Not Survived': '#e74c3c'},
             category_orders={'Class_Name': ['First Class', 'Second Class', 'Third Class']})
fig.update_layout(height=500, hovermode='closest')
fig.write_html('04_fare_distribution.html')
fig.show()
print("Saved: 04_fare_distribution.html")

### 5. Gender-Class Heatmap

In [ ]:
# Survival rate heatmap by gender and class
heatmap_data = df.groupby(['Gender', 'Class_Name'])['Survived'].mean().unstack()
heatmap_data = heatmap_data[['First Class', 'Second Class', 'Third Class']] * 100

fig = px.imshow(heatmap_data, 
                 labels=dict(x='Passenger Class', y='Gender', color='Survival Rate (%)'),
                 title='Survival Rate (%) by Gender and Class',
                 color_continuous_scale='RdYlGn', aspect='auto',
                 text_auto='.1f')
fig.update_layout(height=400)
fig.write_html('05_gender_class_heatmap.html')
fig.show()
print("Saved: 05_gender_class_heatmap.html")

### 6. Embarked Port Analysis

In [ ]:
# Survival by embarkation port
port_data = pd.DataFrame([
    {'Port': 'Cherbourg (C)', 'Status': 'Survived', 'Count': len(df[(df['Port_Name']=='Cherbourg') & (df['Survived']==1)])},
    {'Port': 'Cherbourg (C)', 'Status': 'Did Not Survive', 'Count': len(df[(df['Port_Name']=='Cherbourg') & (df['Survived']==0)])},
    {'Port': 'Queenstown (Q)', 'Status': 'Survived', 'Count': len(df[(df['Port_Name']=='Queenstown') & (df['Survived']==1)])},
    {'Port': 'Queenstown (Q)', 'Status': 'Did Not Survive', 'Count': len(df[(df['Port_Name']=='Queenstown') & (df['Survived']==0)])},
    {'Port': 'Southampton (S)', 'Status': 'Survived', 'Count': len(df[(df['Port_Name']=='Southampton') & (df['Survived']==1)])},
    {'Port': 'Southampton (S)', 'Status': 'Did Not Survive', 'Count': len(df[(df['Port_Name']=='Southampton') & (df['Survived']==0)])},
])

fig = px.bar(port_data, x='Port', y='Count', color='Status',
             title='Survival by Embarkation Port', barmode='group',
             color_discrete_map={'Survived': '#2ecc71', 'Did Not Survive': '#e74c3c'},
             labels={'Count': 'Number of Passengers'})
fig.update_layout(hovermode='x unified', height=500)
fig.write_html('06_port_analysis.html')
fig.show()
print("Saved: 06_port_analysis.html")

### 7. Age vs Fare Scatter Plot

In [ ]:
# Age vs Fare scatter plot
fig = px.scatter(df, x='Age', y='Fare', color='Survived_Label', size='Pclass',
                 title='Age vs Fare (bubble size = passenger class)',
                 labels={'Age': 'Age (years)', 'Fare': 'Ticket Fare (£)'},
                 color_discrete_map={'Survived': '#2ecc71', 'Not Survived': '#e74c3c'},
                 size_max=15)
fig.update_layout(height=600, hovermode='closest')
fig.write_html('07_age_vs_fare.html')
fig.show()
print("Saved: 07_age_vs_fare.html")

### 8. Family Size Impact

In [ ]:
# Create family size variable
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

family_data = pd.DataFrame([
    {'Family Size': i, 'Survival Rate': df[df['FamilySize']==i]['Survived'].mean()*100, 'Count': len(df[df['FamilySize']==i])}
    for i in sorted(df['FamilySize'].unique())
])

fig = go.Figure()
fig.add_trace(go.Bar(x=family_data['Family Size'], y=family_data['Survival Rate'],
                     name='Survival Rate (%)',
                     marker_color='#3498db',
                     yaxis='y',
                     text=family_data['Survival Rate'].round(1),
                     textposition='outside'))
fig.add_trace(go.Scatter(x=family_data['Family Size'], y=family_data['Count'],
                        name='Passenger Count',
                        mode='lines+markers',
                        yaxis='y2',
                        line=dict(color='#e74c3c', width=3),
                        marker=dict(size=10)))

fig.update_layout(
    title='Survival Rate vs Family Size',
    xaxis=dict(title='Family Size (including self)'),
    yaxis=dict(title='Survival Rate (%)', side='left'),
    yaxis2=dict(title='Number of Passengers', overlaying='y', side='right'),
    height=500,
    hovermode='x unified',
    showlegend=True
)
fig.write_html('08_family_size_impact.html')
fig.show()
print("Saved: 08_family_size_impact.html")

### 9. Pie Chart - Survival Overview

In [ ]:
# Survival pie chart
survival_counts = df['Survived_Label'].value_counts()

fig = px.pie(values=survival_counts.values, names=survival_counts.index,
             title='Overall Survival Distribution',
             color_discrete_map={'Survived': '#2ecc71', 'Not Survived': '#e74c3c'},
             labels={'Survived': 'Survived', 'Not Survived': 'Did Not Survive'})
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.write_html('09_survival_overview.html')
fig.show()
print("Saved: 09_survival_overview.html")

---
## Summary Statistics for Dashboard

In [ ]:
# Create summary statistics
summary_stats = {
    'Total Passengers': len(df),
    'Survivors': df['Survived'].sum(),
    'Non-Survivors': len(df) - df['Survived'].sum(),
    'Survival Rate (%)': df['Survived'].mean() * 100,
    'Average Age': df['Age'].mean(),
    'Average Fare': df['Fare'].mean(),
    'Males': len(df[df['Gender']=='Male']),
    'Females': len(df[df['Gender']=='Female']),
    'Female Survival Rate (%)': df[df['Gender']=='Female']['Survived'].mean() * 100,
    'Male Survival Rate (%)': df[df['Gender']=='Male']['Survived'].mean() * 100,
}

print("\n" + "="*60)
print("DASHBOARD SUMMARY STATISTICS")
print("="*60)
for key, value in summary_stats.items():
    if '%' in key:
        print(f"{key:.<40} {value:.2f}%")
    elif isinstance(value, float):
        print(f"{key:.<40} {value:.2f}")
    else:
        print(f"{key:.<40} {value}")
print("="*60)